In [14]:
!pip install -q onnxruntime panns_inference

In [15]:
import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import librosa
import torch
import onnxruntime as ort
from panns_inference import AudioTagging

search_pattern = '/content/drive/MyDrive/**/target_birds_manifest.csv'
manifest_files = glob.glob(search_pattern, recursive=True)
if manifest_files:
    manifest_path = manifest_files[0]
    base_dir = os.path.dirname(os.path.dirname(os.path.dirname(manifest_path)))
else:
    manifest_path = '/content/drive/MyDrive/TUGAS AKHIR/data/manifests/target_birds_manifest.csv'
    base_dir = '/content/drive/MyDrive/TUGAS AKHIR'
split_path = os.path.join(os.path.dirname(manifest_path), 'dataset_split.csv')
ckpt_dir = os.path.join(base_dir, "checkpoints")
if not os.path.exists(ckpt_dir):
    ckpt_dir = "/content/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
print(f"[PATH] Root direktori : {base_dir}")
print(f"[PATH] Folder model   : {ckpt_dir}")
# Unduh otomatis checkpoint jika belum ada di Drive/Colab
r1_ckpt = os.path.join(ckpt_dir, "Cnn14_mAP=0.431.pth")
r2_ckpt = os.path.join(ckpt_dir, "BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx")
if not os.path.exists(r1_ckpt):
    print("[DOWNLOAD] Mengunduh bobot R1 (PANNs CNN14 ~327MB)...")
    url_r1 = "https://huggingface.co/thelou1s/panns-inference/resolve/main/Cnn14_mAP%3D0.431.pth"
    urllib.request.urlretrieve(url_r1, r1_ckpt)
    print("[DOWNLOAD] Selesai unduh R1.")
if not os.path.exists(r2_ckpt):
    print("[DOWNLOAD] Mengunduh bobot R2 (BirdNET ONNX ~20MB)...")
    url_r2 = "https://huggingface.co/biodiversica/BirdNET-onnx-backbone/resolve/main/model_backbone.onnx"
    urllib.request.urlretrieve(url_r2, r2_ckpt)
    print("[DOWNLOAD] Selesai unduh R2.")
# Inisialisasi Model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[DEVICE] Perangkat komputasi : {device.upper()}")
print("[MODEL] Memuat model R1 (PANNs CNN14)...")
panns_model = AudioTagging(checkpoint_path=r1_ckpt, device=device)
print("[MODEL] Memuat model R2 (BirdNET ONNX Backbone)...")
ort_providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device == "cuda" else ["CPUExecutionProvider"]
birdnet_session = ort.InferenceSession(r2_ckpt, providers=ort_providers)
print("[MODEL] Seluruh model pralatih berhasil diinisialisasi.")

# Load dataset
df_manifest = pd.read_csv(manifest_path)
if os.path.exists(split_path):
    df_split = pd.read_csv(split_path)
else:
    df_split = df_manifest.copy()
    df_split['split_role'] = 'gallery'
# Normalisasi penamaan peran query (kompatibel query_clean / clean_query)
df_split['split_role'] = df_split['split_role'].replace({'clean_query': 'query_clean'})
print(f"Total rekaman termuat : {len(df_split)} file")
print("Distribusi partisi   :")
for role, cnt in df_split['split_role'].value_counts().items():
    print(f"   -> {role:12s} : {cnt} rekaman")

[PATH] Root direktori : /content/drive/MyDrive/TUGAS AKHIR
[PATH] Folder model   : /content/drive/MyDrive/TUGAS AKHIR/checkpoints
[DEVICE] Perangkat komputasi : CPU
[MODEL] Memuat model R1 (PANNs CNN14)...
Checkpoint path: /content/drive/MyDrive/TUGAS AKHIR/checkpoints/Cnn14_mAP=0.431.pth
Using CPU.
[MODEL] Memuat model R2 (BirdNET ONNX Backbone)...
[MODEL] Seluruh model pralatih berhasil diinisialisasi.
Total rekaman termuat : 162 file
Distribusi partisi   :
   -> gallery      : 136 rekaman
   -> query_clean  : 14 rekaman
   -> calibration  : 12 rekaman


In [16]:
print("\n" + "-" * 75)
print("[AUDIT] AUDIT SPLIT LEAKAGE & PEMILIHAN SUBSET 20 AUDIO")
print("-" * 75)
df_split = pd.read_csv(split_path)
df_split['split_role'] = df_split['split_role'].replace({'clean_query': 'query_clean'})
def resolve_path(rel_path):
    p1 = os.path.join(base_dir, rel_path)
    if os.path.exists(p1): return p1
    p2 = os.path.join(base_dir, "data", rel_path)
    if os.path.exists(p2): return p2
    return p1
df_split['full_path'] = df_split['file_path'].apply(resolve_path)
df_valid = df_split[df_split['full_path'].apply(os.path.exists)].copy()
# Audit overlap recording_id dan file_path seluruh dataset
all_g_ids = set(df_valid[df_valid['split_role'] == 'gallery']['id'])
all_q_ids = set(df_valid[df_valid['split_role'] == 'query_clean']['id'])
id_overlap = all_g_ids.intersection(all_q_ids)
all_g_paths = set(df_valid[df_valid['split_role'] == 'gallery']['file_path'])
all_q_paths = set(df_valid[df_valid['split_role'] == 'query_clean']['file_path'])
path_overlap = all_g_paths.intersection(all_q_paths)
print(f"[AUDIT] Global ID Overlap (Gallery vs Query)   : {len(id_overlap)} -> {'LOLOS (Bebas Kebocoran)' if len(id_overlap)==0 else 'GAGAL (Bocor)'}")
print(f"[AUDIT] Global Path Overlap (Gallery vs Query) : {len(path_overlap)} -> {'LOLOS (Bebas Kebocoran)' if len(path_overlap)==0 else 'GAGAL (Bocor)'}")
# Ambil subset 10 spesies, masing-masing 1 Query dan 1 Gallery (Total 20 file)
target_species = [sp for sp in df_valid['species_key'].unique()
                  if (len(df_valid[(df_valid['species_key'] == sp) & (df_valid['split_role'] == 'query_clean')]) >= 1 and
                      len(df_valid[(df_valid['species_key'] == sp) & (df_valid['split_role'] == 'gallery')]) >= 1)][:10]
subset_q_rows = []
subset_g_rows = []
for sp in target_species:
    q_row = df_valid[(df_valid['species_key'] == sp) & (df_valid['split_role'] == 'query_clean')].iloc[0]
    g_row = df_valid[(df_valid['species_key'] == sp) & (df_valid['split_role'] == 'gallery')].iloc[0]
    subset_q_rows.append(q_row)
    subset_g_rows.append(g_row)
subset_q_df = pd.DataFrame(subset_q_rows)
subset_g_df = pd.DataFrame(subset_g_rows)
sub_id_leak = set(subset_q_df['id']).intersection(set(subset_g_df['id']))
print(f"[SUBSET] Jumlah data terpilih                  : 10 Query & 10 Gallery (Total: {len(subset_q_df)+len(subset_g_df)} berkas)")
print(f"[SUBSET] Overlap ID pada subset                : {len(sub_id_leak)} -> LOLOS")


---------------------------------------------------------------------------
[AUDIT] AUDIT SPLIT LEAKAGE & PEMILIHAN SUBSET 20 AUDIO
---------------------------------------------------------------------------
[AUDIT] Global ID Overlap (Gallery vs Query)   : 0 -> LOLOS (Bebas Kebocoran)
[AUDIT] Global Path Overlap (Gallery vs Query) : 0 -> LOLOS (Bebas Kebocoran)
[SUBSET] Jumlah data terpilih                  : 10 Query & 10 Gallery (Total: 20 berkas)
[SUBSET] Overlap ID pada subset                : 0 -> LOLOS


In [17]:
TARGET_SR = 32000
DURATION_SEC = 5.0
TARGET_SAMPLES = int(TARGET_SR * DURATION_SEC) # 160.000 sampel
TARGET_RMS = 0.05
def preprocess_audio_pipeline(path):
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    if len(y) > TARGET_SAMPLES:
        hop = TARGET_SAMPLES // 4
        num_windows = max(1, (len(y) - TARGET_SAMPLES) // hop + 1)
        best_rms, best_start = -1.0, 0
        for i in range(num_windows):
            seg = y[i*hop : i*hop + TARGET_SAMPLES]
            rms = np.sqrt(np.mean(seg**2))
            if rms > best_rms:
                best_rms = rms
                best_start = i*hop
        y = y[best_start : best_start + TARGET_SAMPLES]
    elif len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode='constant')
    y = y[:TARGET_SAMPLES]

    current_rms = np.sqrt(np.mean(y**2))
    if current_rms > 1e-6:
        y = y * (TARGET_RMS / current_rms)
    return np.clip(y, -1.0, 1.0).astype(np.float32)

In [18]:
def extract_r0(y):
    """R0: MFCC 20 coef mean+std = 40-dim (L2-normalized)"""
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=20, n_fft=1024, hop_length=512)
    vec = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
    norm = np.linalg.norm(vec)
    return (vec / max(norm, 1e-8)).astype(np.float32)
def extract_r1(y):
    """R1: PANNs CNN14 intermediate embedding = 2048-dim (L2-normalized)"""
    with torch.no_grad():
        t = torch.as_tensor(y[None, :]).float().to(device)
        _, emb_out = panns_model.inference(t)
        emb = emb_out.cpu().numpy().squeeze() if hasattr(emb_out, "cpu") else np.array(emb_out).squeeze()
    norm = np.linalg.norm(emb)
    return (emb / max(norm, 1e-8)).astype(np.float32)
def extract_r2(y):
    """R2: BirdNET ONNX Backbone embedding = 1024-dim (L2-normalized)"""
    y_48k = librosa.resample(y, orig_sr=TARGET_SR, target_sr=48000)
    window_size = 144000  # 3 detik pada 48kHz
    if len(y_48k) < window_size:
        y_48k = np.pad(y_48k, (0, window_size - len(y_48k)), mode='constant')

    windows = []
    hop_size = 48000
    for start in range(0, len(y_48k) - window_size + 1, hop_size):
        windows.append(y_48k[start : start + window_size])
    if not windows:
        windows.append(y_48k[:window_size])

    batch = np.stack(windows, axis=0).astype(np.float32)
    out = birdnet_session.run(["embedding"], {"INPUT": batch})[0]
    emb = np.mean(out, axis=0)
    norm = np.linalg.norm(emb)
    return (emb / max(norm, 1e-8)).astype(np.float32)
def extract_r3(seed=42):
    """R3: Random Gaussian Control = 40-dim (L2-normalized)"""
    rng = np.random.RandomState(seed)
    vec = rng.randn(40).astype(np.float32)
    return vec / np.linalg.norm(vec)

In [19]:
print("\n" + "-" * 75)
print("[PROCESSING] Mengekstrak fitur R0, R1, R2, R3 pada 20 audio subset...")
print("-" * 75)
embs = {
    'R0': {'Q': [], 'G': []},
    'R1': {'Q': [], 'G': []},
    'R2': {'Q': [], 'G': []},
    'R3': {'Q': [], 'G': []}
}
for i, row in subset_g_df.iterrows():
    y = preprocess_audio_pipeline(row['full_path'])
    embs['R0']['G'].append(extract_r0(y))
    embs['R1']['G'].append(extract_r1(y))
    embs['R2']['G'].append(extract_r2(y))
    embs['R3']['G'].append(extract_r3(seed=int(row['id']) % 100000))
for i, row in subset_q_df.iterrows():
    y = preprocess_audio_pipeline(row['full_path'])
    embs['R0']['Q'].append(extract_r0(y))
    embs['R1']['Q'].append(extract_r1(y))
    embs['R2']['Q'].append(extract_r2(y))
    embs['R3']['Q'].append(extract_r3(seed=int(row['id']) % 100000))
q_labels = subset_q_df['species_key'].tolist()
g_labels = subset_g_df['species_key'].tolist()
# Verifikasi dimensi fitur
print(f"[CHECK] Dimensi Vektor R0 (MFCC)    : {embs['R0']['Q'][0].shape} (Target: 40)")
print(f"[CHECK] Dimensi Vektor R1 (PANNs)   : {embs['R1']['Q'][0].shape} (Target: 2048)")
print(f"[CHECK] Dimensi Vektor R2 (BirdNET) : {embs['R2']['Q'][0].shape} (Target: 1024)")
print(f"[CHECK] Dimensi Vektor R3 (Random)  : {embs['R3']['Q'][0].shape} (Target: 40)")


---------------------------------------------------------------------------
[PROCESSING] Mengekstrak fitur R0, R1, R2, R3 pada 20 audio subset...
---------------------------------------------------------------------------
[CHECK] Dimensi Vektor R0 (MFCC)    : (40,) (Target: 40)
[CHECK] Dimensi Vektor R1 (PANNs)   : (2048,) (Target: 2048)
[CHECK] Dimensi Vektor R2 (BirdNET) : (1024,) (Target: 1024)
[CHECK] Dimensi Vektor R3 (Random)  : (40,) (Target: 40)


In [20]:
print("\n" + "=" * 75)
print("📊 EVALUASI PERFORMA RETRIEVAL PADA SUBSET 20 FILE (10 Query vs 10 Gallery)")
print("=" * 75)
def evaluate_subset(Q_list, G_list):
    Q = np.array(Q_list)
    G = np.array(G_list)
    top1_hits = 0
    ap_list = []

    for i in range(len(Q)):
        # Cosine similarity (karena L2-normalized, cukup dot product)
        sims = np.dot(G, Q[i])
        ranked_idx = np.argsort(-sims)
        ranked_sp = [g_labels[idx] for idx in ranked_idx]

        # Cek Top-1
        if ranked_sp[0] == q_labels[i]:
            top1_hits += 1

        # AP (Average Precision)
        rel = np.array([1 if sp == q_labels[i] else 0 for sp in ranked_sp])
        cum_hits = np.cumsum(rel)
        prec = cum_hits / (np.arange(len(rel)) + 1)
        ap = np.sum(prec * rel) / 1.0  # Ada tepat 1 relevan di gallery subset
        ap_list.append(ap)

    return (top1_hits / len(Q)) * 100.0, float(np.mean(ap_list))
results = {}
for code in ['R0', 'R1', 'R2', 'R3']:
    acc1, map_score = evaluate_subset(embs[code]['Q'], embs[code]['G'])
    results[code] = {'Top-1': acc1, 'mAP': map_score}
print(f"{'Representasi':<28} | {'Dimensi':<10} | {'Top-1 Accuracy':<15} | {'Mean AP':<10}")
print("-" * 75)
rep_names = {
    'R0': 'R0: MFCC Baseline',
    'R1': 'R1: PANNs CNN14',
    'R2': 'R2: BirdNET Backbone',
    'R3': 'R3: Random Control'
}
for code in ['R0', 'R1', 'R2', 'R3']:
    dim = embs[code]['Q'][0].shape[0]
    print(f"{rep_names[code]:<28} | {dim:<10} | {results[code]['Top-1']:6.2f}%         | {results[code]['mAP']:6.4f}")
print("-" * 75)
print(f"{'Peluang Tebak Acak Murni':<28} | {'-':<10} | ~{100.0/len(q_labels):5.2f}%         | ~{1.0/len(q_labels):6.4f}")
print("=" * 75)


📊 EVALUASI PERFORMA RETRIEVAL PADA SUBSET 20 FILE (10 Query vs 10 Gallery)
Representasi                 | Dimensi    | Top-1 Accuracy  | Mean AP   
---------------------------------------------------------------------------
R0: MFCC Baseline            | 40         |  50.00%         | 0.6510
R1: PANNs CNN14              | 2048       |  60.00%         | 0.7292
R2: BirdNET Backbone         | 1024       |  70.00%         | 0.8111
R3: Random Control           | 40         |   0.00%         | 0.1735
---------------------------------------------------------------------------
Peluang Tebak Acak Murni     | -          | ~10.00%         | ~0.1000


In [21]:
r3_acc = results['R3']['Top-1']
r3_map = results['R3']['mAP']
all_passed = all(results[c]['Top-1'] > r3_acc or results[c]['mAP'] > r3_map for c in ['R0', 'R1', 'R2'])
if all_passed and len(id_overlap) == 0:
    print("🎉 STATUS SANITY CHECK LENGKAP: [ PASSED / LOLOS ]")
    print("   1. Parameter Preprocessing: TERBEKUKAN (32kHz, 5s, Mono, RMS 0.05).")
    print("   2. Dimensi Representasi   : SESUAI SPESIFIKASI (R0:40, R1:2048, R2:1024, R3:40).")
    print("   3. Kontrol Acak Terbukti  : R3 (Random) < Seluruh Representasi Nyata (R0, R1, R2).")
    print("   4. Kebocoran Data         : ZERO OVERLAP (Recording ID & Path aman).")
    print("   -> Pipeline E0 sempurna! Siap lanjut ke E1 (Clean Retrieval Seluruh Dataset).")
else:
    print("⚠️ STATUS SANITY CHECK: [ PERLU PENINJAUAN ]")
print("=" * 75)

🎉 STATUS SANITY CHECK LENGKAP: [ PASSED / LOLOS ]
   1. Parameter Preprocessing: TERBEKUKAN (32kHz, 5s, Mono, RMS 0.05).
   2. Dimensi Representasi   : SESUAI SPESIFIKASI (R0:40, R1:2048, R2:1024, R3:40).
   3. Kontrol Acak Terbukti  : R3 (Random) < Seluruh Representasi Nyata (R0, R1, R2).
   4. Kebocoran Data         : ZERO OVERLAP (Recording ID & Path aman).
   -> Pipeline E0 sempurna! Siap lanjut ke E1 (Clean Retrieval Seluruh Dataset).
